In [ ]:
##minimax agent reference code: https://www.geeksforgeeks.org/dsa/finding-optimal-move-in-tic-tac-toe-using-minimax-algorithm-in-game-theory/
##wandb link: https://wandb.ai/kradeero-ohio-university/thesis/runs/kdo880z2
# saved model : tictactoe_dqn_minimax_player1.pth

import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import wandb

class BaseModel:
    def __init__(self, discount_factor, epsilon, e_min, e_max):
        self.discount_factor = discount_factor
        self.epsilon = epsilon
        self.e_min = e_min
        self.e_max = e_max
        pass

class Tictactoe_v0:
    def __init__(self):
        self.board = [0] * 9
        self.wining_position = [[0, 1, 2], [3, 4, 5], [6, 7, 8],
                                [0, 3, 6], [1, 4, 7], [2, 5, 8],
                                [0, 4, 8], [6, 4, 2]]
        self.current_turn = 1
        self.player_mark = 1

    def reset(self, is_human_first):
        self.board = [0] * 9
        self.current_turn = 1
        self.player_mark = 1 if is_human_first else -1
        # print("\n--- New Episode ---")
        # self.render()
        if not is_human_first:
            self.env_act()
            # self.render()
        return self.board.copy()

    def check_win(self):
        dqn_symbol = 1 if self.player_mark == 1 else -1  # DQN is X (1) or O (-1)
        opponent_symbol = -dqn_symbol  # Opponent is O (-1) or X (1)
        for pst in self.wining_position:
            line = [self.board[i] for i in pst]
            if line == [dqn_symbol, dqn_symbol, dqn_symbol]:  # DQN wins
                return 1, True
            elif line == [opponent_symbol, opponent_symbol, opponent_symbol]:  # Opponent wins
                return -1, True
        if 0 not in self.board:
            return 0, True  # Draw
        return 0, False

    def board_to_minimax(self):
        minimax_board = [['_' for _ in range(3)] for _ in range(3)]
        for i in range(9):
            row, col = divmod(i, 3)
            if self.board[i] == 1:
                minimax_board[row][col] = 'x'
            elif self.board[i] == -1:
                minimax_board[row][col] = 'o'
            else:
                minimax_board[row][col] = '_'
        return minimax_board

    def is_moves_left(self, board):
        for i in range(3):
            for j in range(3):
                if board[i][j] == '_':
                    return True
        return False

    def evaluate(self, board, player, opponent):
        for row in range(3):
            if board[row][0] == board[row][1] == board[row][2]:
                if board[row][0] == player:
                    return 10
                elif board[row][0] == opponent:
                    return -10
        for col in range(3):
            if board[0][col] == board[1][col] == board[2][col]:
                if board[0][col] == player:
                    return 10
                elif board[0][col] == opponent:
                    return -10
        if board[0][0] == board[1][1] == board[2][2]:
            if board[0][0] == player:
                return 10
            elif board[0][0] == opponent:
                return -10
        if board[0][2] == board[1][1] == board[2][0]:
            if board[0][2] == player:
                return 10
            elif board[0][2] == opponent:
                return -10
        return 0

    def minimax(self, board, depth, is_max, player, opponent):
        score = self.evaluate(board, player, opponent)
        if score == 10:
            return score - depth
        if score == -10:
            return score + depth
        if not self.is_moves_left(board):
            return 0
        if is_max:
            best = -1000
            for i in range(3):
                for j in range(3):
                    if board[i][j] == '_':
                        board[i][j] = player
                        best = max(best, self.minimax(board, depth + 1, not is_max, player, opponent))
                        board[i][j] = '_'
            return best
        else:
            best = 1000
            for i in range(3):
                for j in range(3):
                    if board[i][j] == '_':
                        board[i][j] = opponent
                        best = min(best, self.minimax(board, depth + 1, not is_max, player, opponent))
                        board[i][j] = '_'
            return best

    def find_best_move(self, board, player, opponent):
        best_val = -1000
        best_moves = []
        for i in range(3):
            for j in range(3):
                if board[i][j] == '_':
                    board[i][j] = player
                    move_val = self.minimax(board, 0, False, player, opponent)
                    board[i][j] = '_'
                    if move_val > best_val:
                        best_moves = [(i, j)]
                        best_val = move_val
                    elif move_val == best_val:
                        best_moves.append((i, j))
        return random.choice(best_moves) if best_moves else (-1, -1)

    def env_act(self):
        minimax_board = self.board_to_minimax()
        player = 'o' if self.current_turn == -1 else 'x'
        opponent = 'x' if player == 'o' else 'o'
        row, col = self.find_best_move(minimax_board, player, opponent)
        if row == -1 and col == -1:
            raise Exception('No valid move found by Minimax')
        action = row * 3 + col
        if self.board[action] != 0:
            raise Exception('Invalid action by Minimax')
        self.board[action] = self.current_turn
        # print(f"Minimax ({'O' if self.current_turn == -1 else 'X'}) plays at position {action}")
        reward, done = self.check_win()
        # print(f"Reward after Minimax move: {reward} (Done: {done})")
        self.current_turn = self.current_turn * -1
        return reward, done

    def step(self, action):
        if self.board[action] != 0:
            raise Exception('Invalid action')
        self.board[action] = self.current_turn
        # print(f"DQN ({'X' if self.current_turn == 1 else 'O'}) plays at position {action}")
        # self.render()
        reward, done = self.check_win()
        # print(f"Reward after DQN move: {reward} (Done: {done})")
        self.current_turn = self.current_turn * -1
        if not done:
            reward, done = self.env_act()
            # self.render()
        return self.board.copy(), reward, done, None

    def render(self):
        symbols = {1: 'X', -1: 'O', 0: ' '}
        print("\nCurrent Board:")
        for i in range(3):
            print(f" {symbols[self.board[i*3]]} | {symbols[self.board[i*3+1]]} | {symbols[self.board[i*3+2]]} ")
            if i < 2: print("-----------")
        print()

class EpsilonGreedy:
    def __init__(self, epsilon):
        self.epsilon = epsilon

    def perform(self, q_value, action_space: list = None):
        prob = np.random.sample()
        if prob <= self.epsilon:
            if action_space is None:
                return np.random.randint(len(q_value))
            return np.random.choice(action_space)
        else:
            if action_space is None:
                return np.argmax(q_value)
            return max([[q_value[a], a] for a in action_space], key=lambda x: x[0])[1]

    def decay(self, decay_value, lower_bound):
        self.epsilon = max(self.epsilon * decay_value, lower_bound)

class ExperienceReplay:
    def __init__(self, e_max: int):
        if e_max <= 0:
            raise ValueError('Invalid value for memory size')
        self.e_max = e_max
        self.memory = list()
        self.index = 0

    def add_experience(self, sample: list):
        if len(sample) != 5:
            raise Exception('Invalid sample')
        if len(self.memory) < self.e_max:
            self.memory.append(sample)
        else:
            self.memory[self.index] = sample
        self.index = (self.index + 1) % self.e_max

    def sample_experience(self, sample_size: int, cer_mode: bool):
        samples = random.sample(self.memory, sample_size)
        if cer_mode:
            samples[-1] = self.memory[self.index - 1]
        s_batch, a_batch, r_batch, ns_batch, done_batch = map(np.array, zip(*samples))
        return s_batch, a_batch, r_batch, ns_batch, done_batch

    def get_size(self):
        return len(self.memory)

class DQN(BaseModel):
    def __init__(self, discount_factor: float, epsilon: float, e_min: int, e_max: int):
        super().__init__(discount_factor, epsilon, e_min, e_max)
        self.gamma = discount_factor
        self.epsilon_greedy = EpsilonGreedy(epsilon)
        self.e_min = e_min
        self.exp_replay = ExperienceReplay(e_max)
        self.training_network = nn.Sequential(
            nn.Linear(9, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, 9)
        )
        self.target_network = nn.Sequential(
            nn.Linear(9, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, 9)
        )
        self.target_network.load_state_dict(self.training_network.state_dict())
        self.target_network.eval()
        self.optimizer = optim.RMSprop(self.training_network.parameters(), lr=0.0001)  # Lower LR, use RMSprop
        self.criterion = nn.MSELoss()
        self.cache = list()

    def observe(self, state, action_space: list = None):
        state_tensor = torch.Tensor(np.array([state])).float()
        q_value = self.training_network(state_tensor).detach().numpy().ravel()
        if action_space is not None:
            return max([[q_value[a], a] for a in action_space], key=lambda x: x[0])[1]
        return np.argmax(q_value)

    def observe_on_training(self, state, action_space: list = None) -> int:
        state_tensor = torch.Tensor(np.array([state])).float()
        q_value = self.training_network(state_tensor).detach().numpy().ravel()
        action = self.epsilon_greedy.perform(q_value, action_space)
        self.cache.extend([state, action])
        return action

    def take_reward(self, reward, next_state, done):
        self.cache.extend([reward, next_state, done])
        self.exp_replay.add_experience(self.cache.copy())
        self.cache.clear()

    def train_network(self, sample_size: int, batch_size: int, epochs: int, verbose: int = 2, cer_mode: bool = False):
        if self.exp_replay.get_size() >= self.e_min:
            s_batch, a_batch, r_batch, ns_batch, done_batch = self.exp_replay.sample_experience(sample_size, cer_mode)
            total_loss = 0
            num_batches = (sample_size + batch_size - 1) // batch_size
            for i in range(0, sample_size, batch_size):
                batch_s = s_batch[i:i+batch_size]
                batch_a = a_batch[i:i+batch_size]
                batch_r = r_batch[i:i+batch_size]
                batch_ns = ns_batch[i:i+batch_size]
                batch_done = done_batch[i:i+batch_size]
                batch_states = torch.Tensor(batch_s).float()
                batch_actions = torch.LongTensor(batch_a)
                batch_rewards = torch.Tensor(batch_r).float()
                batch_next_states = torch.Tensor(batch_ns).float()
                batch_done = torch.Tensor(batch_done).float()
                states, q_values = self.replay(batch_states, batch_actions, batch_rewards, batch_next_states, batch_done)
                self.optimizer.zero_grad()
                predictions = self.training_network(states)
                loss = self.criterion(predictions, q_values)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.training_network.parameters(), max_norm=0.1)  # Tighter clipping
                self.optimizer.step()
                total_loss += loss.item()
                # wandb.log({
                #     "Batch Loss": loss.item(),
                #     "Avg Q-Value": predictions.mean().item(),
                #     "Max Q-Value": predictions.max().item()
                # })
            return total_loss / num_batches if num_batches > 0 else 0
        return None

    def replay(self, states, actions, rewards, next_states, terminals):
        # next_actions = self.training_network(next_states).argmax(dim=1)
        # q_values_target = self.target_network(next_states).detach()
        # max_next_q_values = q_values_target[range(len(next_actions)), next_actions]
        max_next_q_values = self.target_network(next_states).detach().max(dim=1)[0]
        q_values = self.training_network(states)
        target_q_values = q_values.clone()

        # Create a tensor for the updated Q-values
        batch_size = states.size(0)
        updates = torch.zeros(batch_size, device=q_values.device)
        for i in range(batch_size):
            if terminals[i]:
                updates[i] = rewards[i]
            else:
                updates[i] = rewards[i] + self.gamma * max_next_q_values[i]
            updates[i] = torch.clamp(updates[i], -2, 2)

        # Update target_q_values using advanced indexing (non-inplace)
        indices = torch.arange(batch_size)
        target_q_values[indices, actions] = updates

        return states, target_q_values

    def update_target_network(self):
        self.target_network.load_state_dict(self.training_network.state_dict())

    def save_model(self, path="tictactoe_dqn_minimax_player2_again.pth"):
        torch.save(self.training_network.state_dict(), path)
        print(f"Model saved to {path}")

wandb.login(key='e21de1f4d4c13b4ba109db92ba20cc946e7da3c5')
wandb.init(project="experiments", name="DQN first VS minimax")

env = Tictactoe_v0()
agent = DQN(0.95, 1, 4096, 1048576)
agent.update_target_network()

num_episodes = 30001
batch_size = 32
epochs = 1
sample_size = 64
target_update_freq = 1000  # Update target network every 100 episodes

total_loss = 0
episode_count_for_avg_loss = 0
win_count = 0
loss_count = 0
draw_count = 0

for episode in range(1, num_episodes+1):
    # print(f"\n--- Episode {episode} ---")
    state = env.reset(is_human_first=True)  # DQN as player 1 (X)
    # print(f"[DEBUG] AI is playing as {'X' if env.player_mark == 1 else 'O'}")
    # print(f"[DEBUG] AI goes {'first' if env.player_mark == 1 else 'second'}")
    done = False
    episode_reward = 0
    while not done:
        action_space = [i for i, val in enumerate(state) if val == 0]
        action = agent.observe_on_training(state, action_space)
        next_state, reward, done, _ = env.step(action)
        agent.take_reward(reward, next_state, done)
        episode_reward = reward
        if agent.exp_replay.get_size() > agent.e_min:
            loss = agent.train_network(sample_size, batch_size, epochs, verbose=0)
            if loss is not None:
                total_loss += loss
        state = next_state
    agent.epsilon_greedy.decay(decay_value=0.995, lower_bound=0.01)
    episode_count_for_avg_loss += 1
    # print(f"Episode Reward: {episode_reward}")
    if episode_reward == 1:
        win_count += 1
    elif episode_reward == -1:
        loss_count += 1
    elif done:
        draw_count += 1
    if episode % target_update_freq == 0:
        agent.update_target_network()
    if episode % 100 == 0:
        avg_loss = total_loss / episode_count_for_avg_loss if episode_count_for_avg_loss > 0 else 0
        print(f"Episode Summary {episode}, Games Won: {win_count}, Games Lost: {loss_count}, Games Drawn: {draw_count}")
        print(f"Episode: {episode}, Avg Loss: {avg_loss:.4f}")
        combined_rate = win_count + draw_count
        wandb.log({
            "Episode": episode,
            "Win Rate": win_count,
            "Loss Rate": loss_count,
            "Draw Rate": draw_count,
            "Win + Draw Rate": combined_rate,
            "Average Loss": avg_loss,
        })
        total_loss = 0
        episode_count_for_avg_loss = 0
        win_count = 0
        loss_count = 0
        draw_count = 0

agent.save_model()
wandb.finish()